# 리포트 ISAC — 드론 탐지·추적 실험 설계 — X410 한 대 · 지향성 안테나 여섯 대 · 매트리스 4E

> ### 한 일
> **시뮬레이션 자료·두 계산 도구의 계약·탐지 코드를 다시 계산해 점검했다(초안 — 하드웨어·파형·실험 설계 절은 이어서 싣는다).**

### 결과
1. 날개 빗살 대비는 빈 하늘 26.8 [^1] ~ 33.4 dB [^2], 지면만 있는 장면 -0.6 [^3] ~ 1.0 dB [^4] 다(등방 안테나, 고립 자세 포함).
2. 지면만 장면의 고립 자세(약 1 %)를 복소 중앙값으로 바꾸면 대비가 27.4 [^5] ~ 32.2 dB [^6] 다.
3. 현재 탐지 코드는 한 CPI 단일 표적 판정까지이고, 추적 필터 이름 정의는 0 개 [^7] 다.
4. PathSolver 는 지면 장면에서 자세당 1.28 s [^8], 우리 커널은 0.058 s [^9] 를 쓴다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 시뮬레이션 자료 | 저장된 샤드를 직접 합쳐 대비·정지/변하는 몫·손잡이 흔들기를 다시 계산 |
| 계산 도구 | 설치된 Sionna RT 2.1.0 과 `src/rcs_sbr.py` 를 CPU 작은 시험으로 호출 |
| 탐지 코드 | 합성 입력으로 생산 함수를 불러 조향·거리축·ECA·CFAR·기준 프레임을 재현 |
| 하드웨어·문헌 | NI·AMD 제원 문서와 다른 연구진의 측정값을 출처와 함께 원장에 기록 |

### 재현

```bash
CUDA_VISIBLE_DEVICES="" /workspace/.venvs/py312/bin/python benchmark/isac_plan_corpus_0915.py
CUDA_VISIBLE_DEVICES="" /workspace/.venvs/py312/bin/python benchmark/isac_plan_engines_0915.py
CUDA_VISIBLE_DEVICES="" /workspace/.venvs/py312/bin/python benchmark/isac_plan_detection_0915.py
CUDA_VISIBLE_DEVICES="" /workspace/.venvs/py312/bin/python benchmark/build_isac_plan_0915.py
```

| | |
|---|---|
| 출력 | `outputs/isac_plan_corpus_0915.json`, `outputs/isac_plan_engines_0915.json`, `outputs/isac_plan_detection_0915.json` |
| 소요 | CPU 한 코어 · 원장 각 수 초~수 분 |

---

## §1. 지금 시뮬레이션 자료가 보여 주는 것

대상은 PathSolver 생산 칸 205 개 [^10] 이다 — 매트리스 4E · 15 m · 3.5 GHz · 광선 4×10⁹ · 자세 8,192 · 확산 켬(`R0D0E0F1`).
자세마다 드론 날개가 도는 몫을 보려고 E 의 평균을 뺀 뒤 스펙트럼을 내고, 날개 반짝임 주파수의 1~12 배음 ±2 칸 평균 전력을 나머지 칸 평균 전력으로 나눈 값을 **날개 빗살 대비**라 부른다.
레벨은 임의 기준의 PathSolver dB 로만 비교한다.

| 장면 | 칸 수 | 빗살 대비 최소 | 최대 | 앙각 |
|---|---|---|---|---|
| 빈 하늘 | 47 [^11] | 26.8 dB [^1] | 33.4 dB [^2] | -90 [^12] ~ 60° [^13] |
| 건물만 | 13 [^14] | 28.2 dB [^15] | 33.4 dB [^16] | -75 [^17] ~ -15° [^18] |
| 지면만 | 17 [^19] | -0.6 dB [^3] | 1.0 dB [^4] | -90 [^20] ~ -15° [^21] |
| 지면+건물 | 20 [^22] | -0.8 dB [^23] | 1.2 dB [^24] | -90 [^25] ~ -15° [^26] |
| 도심 협곡 | 17 [^27] | -0.4 dB [^28] | 1.1 dB [^29] | -75 [^30] ~ -15° [^31] |

![figure 2](../outputs/figures/isac_plan_0915_f2.png)

**그림 2.** 지면이 든 장면에서 날개 빗살 대비는 앙각마다 몇 dB 인가?

### 흔들어 본 손잡이

창·배음 수·칸 폭·바닥 정의를 여섯 가지로 바꿔도, 기본 높이의 지면이 든 장면 최대 대비(고립 자세 포함 값)는 2.0 dB [^32], 빈 하늘 최소 대비는 21.9 dB [^33] 다.
같은 설정 반복 실행 13 묶음 [^34] 에서 대비 차는 최대 0.004 dB [^35] 다.
드론을 지면 위 80 m 로 올린 칸은 6.0 dB [^36] (앙각 −60°) 까지 오른다.

### 지면이 더한 변하는 몫

같은 자세끼리 지면 장면에서 빈 하늘을 빼면, 자세에 따라 변하는 몫이 빈 하늘보다 30.6 [^37] ~ 44.5 dB [^38] 크다(14 쌍 [^39]).
그 몫의 스펙트럼 평탄도는 앙각 −60° 에서 0.558 [^40]로, 같은 길이 흰 잡음의 0.555 [^41] 와 같은 자리에 있다.
광선 수를 10⁹ 이상으로 늘려도 그 수준은 -95.2 [^42] ~ -94.0 dB [^43] 에 머문다(앙각 −60°).

### 고립 자세

**고립 자세** = 8,192 자세의 E 에서 복소 중앙값과의 거리가 그 거리 중앙값의 20 배를 넘는 자세다.
지면만 장면의 등방 안테나 24 칸 [^44] 전부에 고립 자세가 49 [^45] ~ 99 개 [^46] 있고, 변하는 전력의 94.8% [^47] ~ 99.4% [^48] 를 쥔다.
그 칸들은 문턱 50 배에서도 49 [^49] ~ 99 개 [^50] 로 같다.
같은 장면의 조준 안테나 4 칸 [^51] 중 고립 자세가 있는 칸은 1 칸 [^52] 이고, 그 칸이 쥔 몫은 62.3% [^53] 인데, 문턱 50 배에서는 조준 칸 최대가 0 개 [^54] 다 — 고립 자세는 안테나 조건까지 넓혀 읽지 않는다.
그 자세만 복소 중앙값으로 바꾸면 대비가 27.4 [^5] ~ 32.2 dB [^6] 이고, 같은 수의 무작위 자세를 바꾼 대조는 최대 5.9 dB [^55] 다.
반복 실행과 Sionna 판 사이에서 같은 자세가 고립 자세로 잡힌다(Jaccard 최소 0.984 [^56]).
평탄도가 흰 잡음과 같은 자리라는 관찰은 흩어진 충격 몇 개로도 나오므로, 원인은 이 수로 가르지 않는다.

도심 협곡에서는 대체한 대비가 문턱에 따라 움직인다(문턱 50 배에서 최소 4.8 dB [^57]) — 그 칸들은 문턱을 함께 적어 인용한다.
고립 자세에서 어느 경로가 달라지는지는 경로 목록 진단으로 확인한다(DEEP_DROP_0902 의 옛 목록은 실외 전체 장면 두 칸에서 레이다 자신의 지면 반사를 가리켰다).

![figure 1](../outputs/figures/isac_plan_0915_f1.png)

**그림 1.** 앙각 −60° 에서 빈 하늘·지면(등방)·지면(조준)의 변하는 몫 스펙트럼은 어떻게 다른가?

### 조준한 안테나 — 지금까지 도착한 칸

레이다에 3GPP TR 38.901 소자 모형을 달고 드론을 향해 고정 조준했다(최대 감쇠 30 dB).

| 앙각 | 드론 높이 | 등방 대비 | 조준 대비 | 비빗살 바닥 변화 |
|---|---|---|---|---|
| -15° | 20 m | -0.6 dB [^58] | 27.3 dB [^59] | -36.1 dB [^60] |
| -60° | 20 m | -0.0 dB [^61] | 29.3 dB [^62] | -29.0 dB [^63] |
| -60° | 14.5 m | -0.2 dB [^64] | 27.0 dB [^65] | -40.7 dB [^66] |

빈 하늘에서는 조준이 모든 레벨을 16.0 dB [^67] 올린다(소자 최대 이득 왕복 16 dB [^68]).

감쇠 상한 20·50 dB 와 조준 오차를 흔드는 칸 11 개 [^69] 가 아직 큐에 있다. 머리기사 숫자로는 그 칸이 온 뒤에 올린다.
조준 칸의 고립 자세는 기본 높이에서 0 개다 — 이 표만으로는 조준의 몫과 고립 자세의 몫을 가를 수 없고, 두 대비 차이를 안테나 효과로 읽지 않는다.
레이다를 1.5 m 에 단 조준 칸(앙각 −15°)은 고립 자세 53 개 [^70] 가 남는다.
소자 모형은 보유 안테나 여섯 대의 실제 패턴이 아니다 — 제원을 받으면 그 패턴으로 바꿔 다시 계산한다.

### 추적 쪽에서 읽을 것

- 모든 칸은 제자리 비행이다 — 동체는 0 Hz 에 정지 지면과 함께 앉고, 영도플러 제거가 둘을 함께 지운다.
- 자료는 거리 하나·안테나 합 하나라서 거리·각도·이동·열잡음 축이 비어 있다. 추적 정확도는 새 계산(§7)으로 얻는다.
- 창 길이를 줄이면 빈 하늘 대비가 내려간다: 앙각 −15° 에서 자세 1,024 개 19.2 dB [^71], 8,192 개 28.2 dB [^72].

## §3. 두 계산 도구를 추적에 쓸 때 지킬 계약

### PathSolver (Sionna RT 2.1.0 의 경로 계산기)

- 설치본에서 경로 계수 배열의 축은 `[수신기, 수신 안테나, 송신기, 송신 안테나, 경로]` 이고, 합성 배열 기본값에서는 지연·각도가 `[수신기, 송신기, 경로]` 로 안테나 축 없이 나온다. 저장소의 `report15_probe.unpack` 식 평탄화는 계수의 50% [^73] 만 남긴다 — 다중 안테나 작업에서는 축을 그대로 둔다.
- `cir()`·`cfr()` 의 기본 지연 정규화는 링크마다 최소 지연(6.671 ns [^74])을 빼서 공통 위상 125.82° [^75] 를 돌린다. 재추적 구간을 이어 붙이는 추적용 채널은 `normalize_delays=False` 로 받는다.
- 물체 속도 2 m/s 로 낸 도플러는 46.6990 Hz [^76], 2v/λ 는 46.6990 Hz [^77] 다. 한 물체에는 병진 속도 하나만 들어간다 — 회전 날개는 자세마다 장면을 다시 풀거나 조각 메쉬로 나눈다.

- 한 번 푼 뒤 도플러로 외삽한 채널과 다시 푼 채널의 차는 판 0.098 m [^78] 이동(거리 10 m [^79]) 에서 최대 0.98% [^80] 다(CPU 작은 시험).
- 기울기가 2.87° [^81] 를 넘으면 판의 정반사 경로가 1 [^82] 개에서 0 [^83] 개로 사라진다 — 외삽은 경로의 생멸을 따라가지 못하므로 재추적 지점을 둔다.

### 비용 — 궤적 10 초를 자세마다 풀면

| 장면 | 자세당 시간 | 1 kHz · 10 초 | 10 kHz · 10 초 |
|---|---|---|---|
| 빈 하늘 | 0.72 s [^84] | 2.0 h [^85] | 20 h [^86] |
| 지면 | 1.28 s [^8] | 3.6 h [^87] | 36 h [^88] |
| 도심 협곡 | 3.30 s [^89] | 9.2 h [^90] | 92 h [^91] |

시간은 카드를 나눠 쓰는 조건에서 한 프로세스가 잰 벽시계 시간이고, 자세마다 장면을 다시 짓는 시간을 포함한다.
날개 끝 도플러를 겹침 없이 담으려면 제자리 비행에서 2457 Hz [^92], 동체 15 m/s 에서 3158 Hz [^93] 의 자세율이 든다(앙각 −15° 기준).

### 우리 커널 (드론 표면 물리광학 적분)

- 한 번 부를 때 복소수 하나를 낸다. 격자를 드론을 따라 옮기면 λ/8 이동에서 위상이 7.7e-07° [^94], 격자를 고정하면 90.000° [^95] 돈다 — 이동 위상을 쓰려면 격자를 고정한다.
- 반송파를 ±50 MHz 바꿔도 위상 폭이 0.020° [^96] 에 머문다. 거리칸 정보는 출력 밖에 있다 — 지연·도플러·경로 손실은 궤적에서 따로 붙인다.
- 커널 자세당 시간 중앙값은 0.058 s [^9] 다(26 샤드 [^97], 장치 기록 없음).
- 환경 메쉬는 받지 않는다: 격자가 넘긴 메쉬 전체의 상자로 잡혀 지면을 넣으면 격자점이 폭증한다(`benchmark/elevation_sweep_md.py:664-671` 주석의 계산). 평평한 지면은 거울상 법(`--ground`)으로 넣는다 — 그 갈래는 드론 직접·지면 한 번·지면 두 번 경로를 계산하고, 지면 자체의 되돌림은 PathSolver 장면에서 얻는다.

## §4. 탐지 코드 — 한 CPI 판정에서 추적까지

지금 체인은 ECA 직접파 제거 → 거리-도플러 지도 → CA-CFAR 까지, 표적 하나의 한 CPI 판정이다.

| 점검 | 잰 것 | 추적에 옮길 때 할 일 |
|---|---|---|
| 배열 이득 | 조향 벡터 네 가지(참 방향·틀린 방향·영벡터·부호 교대)의 Pd 가 모두 0.552 [^98] | 채널별 복소 IQ 로 각도를 추정한다 |
| 거리축 | 원장의 Rb 만 바꾸면 축이 100 m [^99] 이동, 커널 지연 6 표본 [^100] | 참값 없이 −l_min 으로 축을 세운다 |
| ECA | 0.1 도플러 칸 표적이 -14.8 dB [^101] 남는다 | 느린 표적 전략을 따로 둔다 |
| GPU CFAR | float32 누적합에서 강한 칸 90 dB [^102] 부터 오검출 | 누적합을 float64 로 올린다 |
| 봉우리 | 표적 2 [^103] 개에 CFAR 칸 34 [^104] 개, 돌려주는 봉우리 1 [^105] 개 | 칸을 묶어 검출 목록으로 낸다 |
| 기준 프레임 | 프레임마다 내용이 다르면 손실 -34.3 [^106] ~ -50.9 dB [^107] | 프레임마다 기준을 쓴다 |

기존 결과는 기준 프레임을 한 장 반복해 만든 CPI 라서 마지막 줄의 손실을 겪지 않는다(반복 프레임 대조 0.0 dB [^108]).
리포트 12 의 L1 설정에서 가장 센 칸은 중앙값보다 38.3 dB [^109] 위이고 float32·float64 판정 차는 0 칸 [^110] 이다.

![figure 4](../outputs/figures/isac_plan_0915_f4.png)

**그림 4.** 표적 도플러가 0 에 가까울 때 ECA 뒤에 남는 표적 전력은 몇 dB 인가?

![figure 5](../outputs/figures/isac_plan_0915_f5.png)

**그림 5.** 강한 칸이 배경보다 몇 dB 위일 때 float32 누적합 CFAR 가 오검출을 내기 시작하는가?

### 추적까지 새로 짓는 것

`src/`·`benchmark/` 에서 이름으로 정의된 함수·클래스를 센 결과다(문서 문자열의 언급은 뺀다).

| 부품 | 이름 정의 수 | 쓸 도구 |
|---|---|---|
| 칼만·EKF | 0 [^7] | Stone Soup (`OPENSOURCE.md` 결정) |
| 자료 연관 | 0 [^111] | Stone Soup GNN → JPDA |
| 게이팅 | 0 [^112] | Stone Soup |
| CFAR 칸 묶기 | 0 [^113] | 연결 성분 + 부분칸 보간(`passive_process._subbin` 재사용) |
| 궤적 관리 | 0 [^114] | Stone Soup 확정·삭제 규칙 |
| 추적 지표 | 0 [^115] | GOSPA·연속성 + TR 38.765 지표 |

측정 모델의 야코비안은 `benchmark/verify_observability.py` 의 (거리, 도플러, 각도) 행을 옮겨 쓴다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 큐 0940·0941 의 조준 칸(감쇠 상한·조준 오차·저고도·도심 협곡)을 받아 §1 표를 다시 굽는다 | 조준 안테나 대비가 손잡이를 흔들어도 서는지 | runners/jobs_0940_antenna.txt |
| 안테나 여섯 대의 대역·이득·앞뒤비를 받아 소자 모형을 실제 패턴으로 바꾼다 | 반송파와 시뮬레이션 감쇠 상한 값 | benchmark/report15_probe.py |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 115개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=free_sky].contrast_min_db` | 26.79 |
| [^2] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=free_sky].contrast_max_db` | 33.4 |
| [^3] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_ground].contrast_min_db` | -0.628 |
| [^4] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_ground].contrast_max_db` | 0.954 |
| [^5] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].contrast_replaced_min_db` | 27.45 |
| [^6] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].contrast_replaced_max_db` | 32.24 |
| [^7] | `outputs/isac_plan_detection_0915.json` | `tracking_inventory.rows[component_id=kalman_ekf_ukf].named_definitions_matching` | 0 |
| [^8] | `outputs/isac_plan_engines_0915.json` | `cost.rows[scene=outdoor01_ground,build=rt210,depth=2].median_s_per_pose` | 1.285 |
| [^9] | `outputs/isac_plan_engines_0915.json` | `cost.kernel_cost.median_s_per_pose` | 0.05753 |
| [^10] | `outputs/isac_plan_corpus_0915.json` | `inventory.n_production_complete` | 205 |
| [^11] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=free_sky].n_cells` | 47 |
| [^12] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=free_sky].el_min_deg` | -90 |
| [^13] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=free_sky].el_max_deg` | 60 |
| [^14] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_bldg].n_cells` | 13 |
| [^15] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_bldg].contrast_min_db` | 28.16 |
| [^16] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_bldg].contrast_max_db` | 33.4 |
| [^17] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_bldg].el_min_deg` | -75 |
| [^18] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_bldg].el_max_deg` | -15 |
| [^19] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_ground].n_cells` | 17 |
| [^20] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_ground].el_min_deg` | -90 |
| [^21] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01_ground].el_max_deg` | -15 |
| [^22] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01].n_cells` | 20 |
| [^23] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01].contrast_min_db` | -0.774 |
| [^24] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01].contrast_max_db` | 1.207 |
| [^25] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01].el_min_deg` | -90 |
| [^26] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=outdoor01].el_max_deg` | -15 |
| [^27] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=street_canyon].n_cells` | 17 |
| [^28] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=street_canyon].contrast_min_db` | -0.448 |
| [^29] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=street_canyon].contrast_max_db` | 1.084 |
| [^30] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=street_canyon].el_min_deg` | -75 |
| [^31] | `outputs/isac_plan_corpus_0915.json` | `summary_iso[scene=street_canyon].el_max_deg` | -15 |
| [^32] | `outputs/isac_plan_corpus_0915.json` | `knob_sensitivity.ground_like_max_contrast_default_height_db` | 1.961 |
| [^33] | `outputs/isac_plan_corpus_0915.json` | `knob_sensitivity.free_sky_min_contrast_db` | 21.9 |
| [^34] | `outputs/isac_plan_corpus_0915.json` | `repeats.n_groups` | 13 |
| [^35] | `outputs/isac_plan_corpus_0915.json` | `repeats.max_contrast_spread_db` | 0.004 |
| [^36] | `outputs/isac_plan_corpus_0915.json` | `height.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=80,ant=iso].comb_contrast_db` | 5.962 |
| [^37] | `outputs/isac_plan_corpus_0915.json` | `ground_floor.added_varying_over_free_sky_min_db` | 30.63 |
| [^38] | `outputs/isac_plan_corpus_0915.json` | `ground_floor.added_varying_over_free_sky_max_db` | 44.47 |
| [^39] | `outputs/isac_plan_corpus_0915.json` | `ground_floor.n_matched_iso_pairs` | 14 |
| [^40] | `outputs/isac_plan_corpus_0915.json` | `ground_floor.rows[scene=outdoor01_ground,el_deg=-60,build=2.1.0,depth=2].flatness` | 0.5579 |
| [^41] | `outputs/isac_plan_corpus_0915.json` | `ground_floor.rows[scene=outdoor01_ground,el_deg=-60,build=2.1.0,depth=2].flatness_white_ref` | 0.5555 |
| [^42] | `outputs/isac_plan_corpus_0915.json` | `ray_ladder.varying_min_db_at_or_above_1e9` | -95.2 |
| [^43] | `outputs/isac_plan_corpus_0915.json` | `ray_ladder.varying_max_db_at_or_above_1e9` | -94.03 |
| [^44] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].n_cells` | 24 |
| [^45] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].n_outlier_poses_min` | 49 |
| [^46] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].n_outlier_poses_max` | 99 |
| [^47] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].share_min` | 0.9479 |
| [^48] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].share_max` | 0.9936 |
| [^49] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].n_outlier_poses_f50_min_max[0]` | 49 |
| [^50] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].n_outlier_poses_f50_min_max[1]` | 99 |
| [^51] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=tr38901].n_cells` | 4 |
| [^52] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=tr38901].n_cells_with_outliers` | 1 |
| [^53] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=tr38901].share_max` | 0.6227 |
| [^54] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=tr38901].n_outlier_poses_f50_min_max[1]` | 0 |
| [^55] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].random_control_contrast_max_db` | 5.927 |
| [^56] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=iso].cross_build_jaccard_min` | 0.9839 |
| [^57] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=street_canyon,ant=iso].contrast_replaced_f50_min_max_db[0]` | 4.833 |
| [^58] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-15,env_alt_m=20].contrast_iso_db` | -0.628 |
| [^59] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-15,env_alt_m=20].contrast_ant_db` | 27.33 |
| [^60] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-15,env_alt_m=20].noncomb_delta_db` | -36.12 |
| [^61] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=20].contrast_iso_db` | -0.026 |
| [^62] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=20].contrast_ant_db` | 29.29 |
| [^63] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=20].noncomb_delta_db` | -29.02 |
| [^64] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=14.5].contrast_iso_db` | -0.25 |
| [^65] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=14.5].contrast_ant_db` | 26.96 |
| [^66] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=outdoor01_ground,el_deg=-60,env_alt_m=14.5].noncomb_delta_db` | -40.69 |
| [^67] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.rows[scene=free_sky,el_deg=-60].total_delta_db` | 16.01 |
| [^68] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.code_expectation.two_way_peak_gain_db` | 16 |
| [^69] | `outputs/isac_plan_corpus_0915.json` | `antenna_pairs.n_queued_ant_cells_not_landed_active` | 11 |
| [^70] | `outputs/isac_plan_corpus_0915.json` | `dropout.summary[scene=outdoor01_ground,ant=tr38901].n_outlier_poses_max` | 53 |
| [^71] | `outputs/isac_plan_corpus_0915.json` | `dwell.rows[scene=free_sky,ant=iso,el_deg=-15,window_poses=1024].comb_contrast_db` | 19.25 |
| [^72] | `outputs/isac_plan_corpus_0915.json` | `dwell.rows[scene=free_sky,ant=iso,el_deg=-15,window_poses=8192].comb_contrast_db` | 28.16 |
| [^73] | `outputs/isac_plan_engines_0915.json` | `rt_contract.unpack_style_kept_fraction_of_coefficients` | 0.5 |
| [^74] | `outputs/isac_plan_engines_0915.json` | `rt_contract.normalize_delays.min_tau_ns` | 6.671 |
| [^75] | `outputs/isac_plan_engines_0915.json` | `rt_contract.normalize_delays.phase_rotation_deg_measured` | 125.8 |
| [^76] | `outputs/isac_plan_engines_0915.json` | `rt_doppler.doppler_hz_measured` | 46.7 |
| [^77] | `outputs/isac_plan_engines_0915.json` | `rt_doppler.doppler_hz_expected_2v_over_lambda` | 46.7 |
| [^78] | `outputs/isac_plan_engines_0915.json` | `extrapolation.translation.distance_m` | 0.098 |
| [^79] | `outputs/isac_plan_engines_0915.json` | `extrapolation.translation.range_m` | 10 |
| [^80] | `outputs/isac_plan_engines_0915.json` | `extrapolation.translation.max_rel_diff` | 0.009802 |
| [^81] | `outputs/isac_plan_engines_0915.json` | `extrapolation.rotation_tilt.specular_cutoff_tilt_deg` | 2.866 |
| [^82] | `outputs/isac_plan_engines_0915.json` | `extrapolation.rotation_tilt.n_specular_paths_before` | 1 |
| [^83] | `outputs/isac_plan_engines_0915.json` | `extrapolation.rotation_tilt.n_specular_paths_after` | 0 |
| [^84] | `outputs/isac_plan_engines_0915.json` | `cost.rows[scene=free_sky,build=rt210,depth=2].median_s_per_pose` | 0.72 |
| [^85] | `outputs/isac_plan_engines_0915.json` | `cost.trajectory_rows[scene=free_sky,build=rt210,rate_hz=1000].single_process_wall_hours` | 2 |
| [^86] | `outputs/isac_plan_engines_0915.json` | `cost.trajectory_rows[scene=free_sky,build=rt210,rate_hz=10000].single_process_wall_hours` | 20 |
| [^87] | `outputs/isac_plan_engines_0915.json` | `cost.trajectory_rows[scene=outdoor01_ground,build=rt210,rate_hz=1000].single_process_wall_hours` | 3.568 |
| [^88] | `outputs/isac_plan_engines_0915.json` | `cost.trajectory_rows[scene=outdoor01_ground,build=rt210,rate_hz=10000].single_process_wall_hours` | 35.68 |
| [^89] | `outputs/isac_plan_engines_0915.json` | `cost.rows[scene=sionna-simple_street_canyon,build=rt210,depth=2].median_s_per_pose` | 3.296 |
| [^90] | `outputs/isac_plan_engines_0915.json` | `cost.trajectory_rows[scene=sionna-simple_street_canyon,build=rt210,rate_hz=1000].single_process_wall_hours` | 9.156 |
| [^91] | `outputs/isac_plan_engines_0915.json` | `cost.trajectory_rows[scene=sionna-simple_street_canyon,build=rt210,rate_hz=10000].single_process_wall_hours` | 91.56 |
| [^92] | `outputs/isac_plan_engines_0915.json` | `rotor.required_pose_rate[body_speed_mps=0].required_rate_hz` | 2457 |
| [^93] | `outputs/isac_plan_engines_0915.json` | `rotor.required_pose_rate[body_speed_mps=15].required_rate_hz` | 3158 |
| [^94] | `outputs/isac_plan_engines_0915.json` | `kernel.recentred_phase_deg_for_lambda_over_8_move` | 7.673e-07 |
| [^95] | `outputs/isac_plan_engines_0915.json` | `kernel.frozen_grid_phase_deg` | 90 |
| [^96] | `outputs/isac_plan_engines_0915.json` | `kernel.phase_vs_frequency_measured_span_deg` | 0.01958 |
| [^97] | `outputs/isac_plan_engines_0915.json` | `cost.kernel_cost.n_shards` | 26 |
| [^98] | `outputs/isac_plan_detection_0915.json` | `f1_steering.variants[name=steer_true_az].pd` | 0.5521 |
| [^99] | `outputs/isac_plan_detection_0915.json` | `f2_range_axis.axis_shift_m_when_meta_rb_changes` | 100 |
| [^100] | `outputs/isac_plan_detection_0915.json` | `f2_range_axis.measured_extra_lag_samples` | 6 |
| [^101] | `outputs/isac_plan_detection_0915.json` | `f3_eca.rows[doppler_bin=0.1].retained_db_measured` | -14.81 |
| [^102] | `outputs/isac_plan_detection_0915.json` | `f4_cfar.first_strong_cell_db_with_float32_false_cells` | 90 |
| [^103] | `outputs/isac_plan_detection_0915.json` | `f5_multi_target.n_targets` | 2 |
| [^104] | `outputs/isac_plan_detection_0915.json` | `f5_multi_target.cfar_mask_cells` | 34 |
| [^105] | `outputs/isac_plan_detection_0915.json` | `f5_multi_target.peaks_returned_by_peak_detection` | 1 |
| [^106] | `outputs/isac_plan_detection_0915.json` | `f14_reference.loss_db_max` | -34.28 |
| [^107] | `outputs/isac_plan_detection_0915.json` | `f14_reference.loss_db_min` | -50.87 |
| [^108] | `outputs/isac_plan_detection_0915.json` | `f14_reference.max_abs_repeated_frame_control_db` | 0 |
| [^109] | `outputs/isac_plan_detection_0915.json` | `f4_cfar.report12_l1_check.max_cell_over_median_db` | 38.26 |
| [^110] | `outputs/isac_plan_detection_0915.json` | `f4_cfar.report12_l1_check.mask_mismatches` | 0 |
| [^111] | `outputs/isac_plan_detection_0915.json` | `tracking_inventory.rows[component_id=data_association].named_definitions_matching` | 0 |
| [^112] | `outputs/isac_plan_detection_0915.json` | `tracking_inventory.rows[component_id=gating].named_definitions_matching` | 0 |
| [^113] | `outputs/isac_plan_detection_0915.json` | `tracking_inventory.rows[component_id=cfar_cell_clustering].named_definitions_matching` | 0 |
| [^114] | `outputs/isac_plan_detection_0915.json` | `tracking_inventory.rows[component_id=track_management].named_definitions_matching` | 0 |
| [^115] | `outputs/isac_plan_detection_0915.json` | `tracking_inventory.rows[component_id=tracking_metrics].named_definitions_matching` | 0 |